In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import os
import glob
import io
import base64
from IPython.display import HTML, display

os.makedirs("models", exist_ok=True)


## Function for displaying videos
def show_video(video_folder="videos"):
    mp4list = glob.glob(f'{video_folder}/*.mp4')
    if len(mp4list) > 0:
        mp4 = max(mp4list, key=os.path.getctime)
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay 
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("No vide found")

## Orthogonal initialisation
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

## Actor critic network
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.critic = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 512)),
            nn.Tanh(),
            layer_init(nn.Linear(512, 512)),
            nn.Tanh(),
            layer_init(nn.Linear(512, 1), std=1.0),
        )
        self.actor_mean = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 512)),
            nn.Tanh(),
            layer_init(nn.Linear(512, 512)),
            nn.Tanh(),
            layer_init(nn.Linear(512, act_dim), std=0.01),
        )
        self.actor_logstd = nn.Parameter(torch.ones(1, act_dim) * -0.5)

    ## Predictions
    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None):
        action_mean = self.actor_mean(x)
        action_logstd = self.actor_logstd.expand_as(action_mean)
        action_std = torch.exp(action_logstd)

        dist = Normal(action_mean, action_std)

        if action is None:
            action = dist.sample()

        log_prob = dist.log_prob(action).sum(1)
        entropy = dist.entropy().sum(1)
        value = self.critic(x)

        return action, log_prob, entropy, value.squeeze(1)

class RolloutBuffer:
    def __init__(self, size, obs_dim, act_dim, device):
        self.size = size
        self.device = device
        self.obs = np.zeros((size, obs_dim), dtype=np.float32)
        self.actions = np.zeros((size, act_dim), dtype=np.float32)
        self.log_probs = np.zeros(size, dtype=np.float32)
        self.rewards = np.zeros(size, dtype=np.float32)
        self.dones = np.zeros(size, dtype=np.float32)
        self.values = np.zeros(size, dtype=np.float32)
        self.advantages = np.zeros(size, dtype=np.float32)
        self.returns = np.zeros(size, dtype=np.float32)
        self.ptr = 0
        self.path_start_idx = 0

    def store(self, obs, action, log_prob, reward, done, value):
        assert self.ptr < self.size
        self.obs[self.ptr] = obs
        self.actions[self.ptr] = action
        self.log_probs[self.ptr] = log_prob
        self.rewards[self.ptr] = reward
        self.dones[self.ptr] = done
        self.values[self.ptr] = value
        self.ptr += 1

    ## Computing GAE and returns
    def finish_path(self, last_value, gamma, lam):
        path_slice = slice(self.path_start_idx, self.ptr)
        rewards = np.append(self.rewards[path_slice], last_value)
        values = np.append(self.values[path_slice], last_value)
        
        gae = 0.0
        adv = np.zeros_like(self.rewards[path_slice])
        
        for t in reversed(range(len(rewards) - 1)):
            delta = rewards[t] + gamma * values[t + 1] * (1 - self.dones[path_slice][t]) - values[t]
            gae = delta + gamma * lam * (1 - self.dones[path_slice][t]) * gae
            adv[t] = gae
            
        self.advantages[path_slice] = adv
        self.returns[path_slice] = adv + self.values[path_slice]
        self.path_start_idx = self.ptr

    ## Getting data for training
    def get(self):
        assert self.ptr == self.size
        self.ptr = 0
        self.path_start_idx = 0
        
        adv = self.advantages
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        
        return dict(
            obs=torch.tensor(self.obs, dtype=torch.float32, device=self.device),
            actions=torch.tensor(self.actions, dtype=torch.float32, device=self.device),
            log_probs=torch.tensor(self.log_probs, dtype=torch.float32, device=self.device),
            advantages=torch.tensor(adv, dtype=torch.float32, device=self.device),
            returns=torch.tensor(self.returns, dtype=torch.float32, device=self.device),
            values=torch.tensor(self.values, dtype=torch.float32, device=self.device),
        )

class PPOAgent:
    def __init__(
        self,
        env_id="BipedalWalker-v3",
        total_timesteps=5_000_000,
        rollout_steps=4096,
        gamma=0.99,
        lam=0.95,
        clip_eps=0.2,
        learning_rate=2.5e-4,
        train_epochs=10,
        minibatch_size=512,
        vf_coef=0.5,
        ent_coef=0.00,
        render_freq=50, 
        device="cpu",
    ):
        self.env_id = env_id
        self.total_timesteps = total_timesteps
        self.rollout_steps = rollout_steps
        self.gamma = gamma
        self.lam = lam
        self.clip_eps = clip_eps
        self.learning_rate = learning_rate
        self.train_epochs = train_epochs
        self.minibatch_size = minibatch_size
        self.vf_coef = vf_coef
        self.ent_coef = ent_coef
        self.render_freq = render_freq
        self.device = device

        self.env = gym.make(env_id)
        self.env = gym.wrappers.RecordEpisodeStatistics(self.env)
        self.env = gym.wrappers.ClipAction(self.env)
        
        self.env = gym.wrappers.NormalizeObservation(self.env)
        self.env = gym.wrappers.TransformObservation(self.env, lambda obs: np.clip(obs, -10, 10), self.env.observation_space)

        self.obs_dim = self.env.observation_space.shape[0]
        self.act_dim = self.env.action_space.shape[0]

        self.ac = ActorCritic(self.obs_dim, self.act_dim).to(device)
        self.optimizer = optim.Adam(self.ac.parameters(), lr=self.learning_rate, eps=1e-5)
        
        self.num_updates = total_timesteps // rollout_steps
        self.lr_scheduler = torch.optim.lr_scheduler.LinearLR(
            self.optimizer, start_factor=1.0, end_factor=0.0, total_iters=self.num_updates
        )

        self.buffer = RolloutBuffer(self.rollout_steps, self.obs_dim, self.act_dim, self.device)

    def visualize_agent(self, update_count):
        print(f"\n--- Visualizing Agent at Update {update_count} ---")
        
        vis_env = gym.make(self.env_id, render_mode="rgb_array")
        
        vis_env = gym.wrappers.RecordVideo(
            vis_env, 
            video_folder="videos", 
            name_prefix=f"update_{update_count}",
            disable_logger=True
        )
        
        vis_env = gym.wrappers.ClipAction(vis_env)
        
        vis_norm = gym.wrappers.NormalizeObservation(vis_env)
        try:
            vis_norm.obs_rms = self.env.get_wrapper_attr('obs_rms')
        except AttributeError:
            pass

        vis_env = gym.wrappers.TransformObservation(vis_norm, lambda obs: np.clip(obs, -10, 10), vis_env.observation_space)
        
        obs, _ = vis_env.reset()
        ret = 0
        
        while True:
            obs_tensor = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
            with torch.no_grad():
                action = self.ac.actor_mean(obs_tensor).squeeze(0).cpu().numpy()
            
            obs, reward, terminated, truncated, _ = vis_env.step(action)
            ret += reward
            if terminated or truncated:
                break
        
        vis_env.close()
        print(f"Visualization finished with return: {ret:.2f}")
        show_video("videos")

    def train(self):
        obs, _ = self.env.reset()
        timesteps_collected = 0
        update_count = 0

        while timesteps_collected < self.total_timesteps:
            for _ in range(self.rollout_steps):
                obs_tensor = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
                with torch.no_grad():
                    action, log_prob, _, value = self.ac.get_action_and_value(obs_tensor)

                action = action.cpu().numpy().squeeze(0)
                log_prob = log_prob.item()
                value = value.item()

                next_obs, reward, terminated, truncated, infos = self.env.step(action)
                done = terminated or truncated

                self.buffer.store(obs, action, log_prob, reward, done, value)
                timesteps_collected += 1
                obs = next_obs

                if "episode" in infos:
                    print(f"Update {update_count} | Steps: {timesteps_collected} | Return: {infos['episode']['r']:.2f}")

                if done:
                    if truncated:
                        last_val_obs = torch.tensor(next_obs, dtype=torch.float32, device=self.device).unsqueeze(0)
                        with torch.no_grad():
                            last_value = self.ac.get_value(last_val_obs).item()
                    else:
                        last_value = 0
                    
                    self.buffer.finish_path(last_value=last_value, gamma=self.gamma, lam=self.lam)
                    obs, _ = self.env.reset()
                
                if timesteps_collected >= self.total_timesteps:
                    break

            if not done:
                obs_tensor = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
                with torch.no_grad():
                    last_value = self.ac.get_value(obs_tensor).item()
                self.buffer.finish_path(last_value=last_value, gamma=self.gamma, lam=self.lam)

            if self.buffer.ptr == self.buffer.size:
                data = self.buffer.get()
                self._update(data)
                self.lr_scheduler.step()
                update_count += 1
            
            if update_count > 0 and update_count % self.render_freq == 0:
                self.visualize_agent(update_count)

        self.env.close()

    ## PPO updates
    def _update(self, data):
        obs = data["obs"]
        actions = data["actions"]
        old_log_probs = data["log_probs"]
        advantages = data["advantages"]
        returns = data["returns"]

        batch_size = len(obs)
        inds = np.arange(batch_size)

        for _ in range(self.train_epochs):
            np.random.shuffle(inds)
            for start in range(0, batch_size, self.minibatch_size):
                end = start + self.minibatch_size
                mb_inds = inds[start:end]

                mb_obs = obs[mb_inds]
                mb_actions = actions[mb_inds]
                mb_old_log_probs = old_log_probs[mb_inds]
                mb_adv = advantages[mb_inds]
                mb_returns = returns[mb_inds]

                _, new_log_probs, entropy, values = self.ac.get_action_and_value(mb_obs, mb_actions)

                ratio = torch.exp(new_log_probs - mb_old_log_probs)
                surr1 = ratio * mb_adv
                surr2 = torch.clamp(ratio, 1.0 - self.clip_eps, 1.0 + self.clip_eps) * mb_adv

                actor_loss = -torch.min(surr1, surr2).mean()
                critic_loss = 0.5 * ((values - mb_returns) ** 2).mean()
                entropy_loss = entropy.mean()

                loss = actor_loss + self.vf_coef * critic_loss - self.ent_coef * entropy_loss

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.ac.parameters(), max_norm=0.5)
                self.optimizer.step()

    def save(self, path="ppo_bipedal.pt"):
        torch.save(self.ac.state_dict(), path)

if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    agent = PPOAgent(
        env_id="BipedalWalker-v3",
        total_timesteps=2_000_000,
        rollout_steps=4096,
        render_freq=25,
        device=device,
    )

    agent.train()
    agent.save("ppo_bipedal_final.pt")

    import pickle
    with open("obs_stats.pkl", "wb") as f:
        pickle.dump(agent.env.get_wrapper_attr('obs_rms'), f)